# EDA — Preços quebrados no catálogo (P3)

**Objetivo:** entender se a Silver de produtos e categorias serve para prever o preço esperado e achar preços fora do padrão.

**Perguntas do EDA**
1. Quantos produtos, categorias e marcas existem?
2. Existem `preco_custo` e uma coluna de tamanho na Silver?
3. Como é a distribuição de `preco_lista` (zeros, negativos, extremos)?
4. O preço varia por subcategoria, marca e tamanho de forma previsível?
5. Quantos produtos as regras de preço já marcam (rótulo fraco)?
6. Existe margem negativa (`preco_lista` < `preco_custo`)?

**Entradas (lidas do lake, camada Silver)**
- `squad1/silver/ecommerce_produtos`: uma linha por observação de preço (micro-lote × SKU).
- `squad1/silver/ecommerce_categorias`: hierarquia de categorias (raiz e subcategoria).

**Saídas**
- Nenhuma tabela é gravada. Este notebook só analisa e termina com as conclusões da EDA, que orientam o notebook 08 (engenharia de features).

**Como rodar**
- Use "Executar tudo", do começo ao fim. Algumas células usam variáveis criadas nas anteriores.
- É preciso ter o arquivo `.env` na mesma pasta, com `ADLS_CLIENT_ID`, `ADLS_CLIENT_SECRET` e `ADLS_TENANT_ID`. Ele não vai para o Git porque guarda credenciais.

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F

load_dotenv(".env")

STORAGE_ACCOUNT = "internshipdatalake"
lakehouse_base = f"abfss://squad1@{STORAGE_ACCOUNT}.dfs.core.windows.net"

adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": os.getenv("ADLS_CLIENT_ID"),
    "fs.azure.account.oauth2.client.secret": os.getenv("ADLS_CLIENT_SECRET"),
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}

def ler_delta_spark(caminho):
    return spark.read.format("delta").options(**adls_options).load(caminho)

silver_produtos_path = f"{lakehouse_base}/silver/ecommerce_produtos"
silver_categorias_path = f"{lakehouse_base}/silver/ecommerce_categorias"

df_prod = ler_delta_spark(silver_produtos_path)
df_cat = ler_delta_spark(silver_categorias_path)

print("Produtos:  ", df_prod.count(), "linhas |", len(df_prod.columns), "colunas")
print("Categorias:", df_cat.count(), "linhas |", len(df_cat.columns), "colunas")

print("\n=== Schema: produtos ===")
df_prod.printSchema()

print("=== Schema: categorias ===")
df_cat.printSchema()

In [0]:
# ------------------------------------------------------------
# Conhecendo as tabelas
# Mostra exemplos de linhas, as unidades de medida e a contagem
# de marcas e categorias, para entender o que existe nos dados.
# ------------------------------------------------------------

# Amostra de produtos: só as colunas que importam para o preço.
# truncate=45 corta textos longos em 45 caracteres para caber na tela.
print("=== Amostra de produtos ===")
(df_prod
    .select("sku", "nome_produto", "descricao", "preco_lista", "unidade_medida", "nome_marca", "id_categoria", "id_categoria_pai")
    .show(12, truncate=45))

# Quantas linhas há de cada unidade de medida (da mais frequente para a menos).
print("=== Unidades de medida ===")
df_prod.groupBy("unidade_medida").count().orderBy(F.desc("count")).show(20, truncate=False)

# Quantos valores distintos existem em marca, subcategoria e categoria raiz.
# (id_categoria é a subcategoria do produto; id_categoria_pai é a categoria raiz.)
print("Marcas distintas:", df_prod.select("nome_marca").distinct().count())
print("Categorias distintas em produtos:", df_prod.select("id_categoria").distinct().count())
print("Pais (categoria raiz) distintos em produtos:", df_prod.select("id_categoria_pai").distinct().count())

# Amostra da tabela de categorias.
print("\n=== Amostra de categorias ===")
(df_cat
    .select("id_categoria", "nome_categoria", "id_categoria_pai", "nome_categoria_pai", "tipo_categoria")
    .show(12, truncate=40))

# A hierarquia se repete a cada micro-lote, por isso os totais são grandes:
# tipo "Categoria" = raiz; tipo "Subcategoria" = nível abaixo.
print("=== tipo_categoria ===")
df_cat.groupBy("tipo_categoria").count().show(truncate=False)

In [0]:
# ------------------------------------------------------------
# Qualidade dos dados
# Conta linhas, SKUs e micro-lotes, procura valores nulos e mostra
# quantas linhas falham em cada regra de qualidade da Etapa 1.
# ------------------------------------------------------------

# Guardamos o total de linhas para calcular percentuais mais adiante.
total = df_prod.count()
print("Linhas:", total)
print("SKUs distintos:", df_prod.select("sku").distinct().count())
# Cada arquivo (bronze_source_file) é um micro-lote; esperamos 365.
print("Micro-lotes (arquivos) distintos:", df_prod.select("bronze_source_file").distinct().count())

# Nulos por coluna: soma 1 para cada valor nulo e 0 para cada valor preenchido.
print("\n=== Nulos por coluna ===")
colunas_dado = ["id_categoria", "sku", "nome_produto", "descricao", "preco_lista",
                "unidade_medida", "nome_marca", "is_ativo", "id_categoria_pai"]
nulos = df_prod.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas_dado]).collect()[0]
for c in colunas_dado:
    print(f"{c}: {nulos[c]} nulos")

# As regras da Etapa 1 são colunas booleanas (regra_*): True = passou, False = falhou.
# Aqui contamos, para cada regra, quantas linhas FALHARAM.
# Para o P3 interessam a regra 02 (preço positivo) e a 07 (preço dentro do limite),
# que formam o rótulo fraco. A regra 09 mede vendas, e não preço.
print("\n=== Linhas que FALHAM em cada regra ===")
colunas_regra = sorted([c for c in df_prod.columns if c.startswith("regra_")])
falhas = df_prod.select(
    [F.sum(F.when(F.col(c) == False, 1).otherwise(0)).alias(c) for c in colunas_regra]
).collect()[0]
for c in colunas_regra:
    print(f"{c}: {falhas[c]} falhas ({falhas[c] / total * 100:.2f}%)")

In [0]:
# ------------------------------------------------------------
# Grão da tabela
# Descobre o que significa UMA linha: quantas linhas tem cada micro-lote,
# se um SKU se repete dentro do mesmo micro-lote e quantas vezes cada SKU
# aparece. Isso define como o preço deve ser modelado.
# ------------------------------------------------------------

# 5.1 Quantas linhas tem cada micro-lote?
# Se todos tiverem o mesmo número, os micro-lotes têm tamanho fixo.
print("=== Linhas por micro-lote ===")
por_lote = df_prod.groupBy("bronze_source_file").count()
por_lote.agg(
    F.min("count").alias("minimo"),
    F.max("count").alias("maximo"),
    F.round(F.avg("count"), 1).alias("media"),
).show()

# 5.2 O mesmo SKU aparece mais de uma vez DENTRO do mesmo micro-lote?
# Se o número de pares (micro-lote, SKU) distintos for igual ao total de linhas,
# cada SKU aparece no máximo uma vez por micro-lote.
pares = df_prod.select("bronze_source_file", "sku").distinct().count()
print("Pares (micro-lote, SKU) distintos:", pares, "de", df_prod.count(), "linhas")

# 5.3 Quantas vezes cada SKU aparece, e quantos preços diferentes ele tem?
# Também confere se categoria e marca são fixas por SKU (n_categorias e n_marcas = 1).
por_sku = df_prod.groupBy("sku").agg(
    F.count("*").alias("n_linhas"),
    F.countDistinct("preco_lista").alias("n_precos"),
    F.countDistinct("id_categoria").alias("n_categorias"),
    F.countDistinct("nome_marca").alias("n_marcas"),
    F.min("preco_lista").alias("preco_min"),
    F.max("preco_lista").alias("preco_max"),
)

print("=== Resumo por SKU ===")
por_sku.agg(
    F.min("n_linhas").alias("linhas_min"),
    F.max("n_linhas").alias("linhas_max"),
    F.round(F.avg("n_linhas"), 1).alias("linhas_media"),
    F.round(F.avg("n_precos"), 1).alias("precos_distintos_media"),
    F.max("n_categorias").alias("max_categorias_por_sku"),
    F.max("n_marcas").alias("max_marcas_por_sku"),
).show()

# Conclusão desta célula: uma linha é a observação do preço de um SKU em um
# micro-lote. O preço de um mesmo SKU muda de uma observação para outra.
print("=== 5 SKUs que mais aparecem ===")
por_sku.orderBy(F.desc("n_linhas")).show(5, truncate=False)

In [0]:
# ------------------------------------------------------------
# Distribuição do preço (preco_lista)
# Mostra como o preço se distribui, quantos são zero, negativos ou muito altos,
# qual é o limite da regra 07 e quanto cada preço se afasta do nível normal
# do próprio SKU.
# ATENÇÃO: esta célula cria positivos, mediana_sku e df_razao,
# que as células seguintes usam. Rode sempre com "Executar tudo".
# ------------------------------------------------------------

# 6.1 Visão geral de preco_lista.
# A mediana (50%) é puxada pelos erros, e o mínimo e o máximo mostram os valores extremos.
print("=== Estatísticas gerais de preco_lista ===")
df_prod.select("preco_lista").summary("count", "min", "25%", "50%", "75%", "max", "mean", "stddev").show()

# Quantos preços são zero, negativos ou altos. Os faixas de 100, 500 e 1000 são
# só para dar uma ideia do tamanho da cauda; não são limites de regra.
print("=== Quantos preços zero, negativos e altos ===")
df_prod.select(
    F.sum((F.col("preco_lista") == 0).cast("int")).alias("zero"),
    F.sum((F.col("preco_lista") < 0).cast("int")).alias("negativos"),
    F.sum((F.col("preco_lista") > 100).cast("int")).alias("acima_100"),
    F.sum((F.col("preco_lista") > 500).cast("int")).alias("acima_500"),
    F.sum((F.col("preco_lista") > 1000).cast("int")).alias("acima_1000"),
).show()

# 6.2 Qual é o limite da regra 07?
# Compara o menor e o maior preço das linhas que passaram e das que falharam na regra.
# O maior preço das linhas aprovadas mostra o limite (cerca de R$ 5.000).
print("=== Menor e maior preço por resultado da regra 07 ===")
df_prod.groupBy("regra_07_preco_dentro_limite").agg(
    F.count("*").alias("linhas"),
    F.min("preco_lista").alias("menor_preco"),
    F.max("preco_lista").alias("maior_preco"),
).show()

# 6.3 Razão entre o preço e a mediana do próprio SKU (só preços positivos).
# A mediana do SKU é o "nível normal" dele. razao = 1 é um preço normal;
# razao = 10 é um preço dez vezes maior que o normal.
# Só preços positivos entram, para que zeros e negativos não distorçam a mediana.
positivos = df_prod.filter(F.col("preco_lista") > 0)
mediana_sku = positivos.groupBy("sku").agg(
    F.percentile_approx(F.col("preco_lista").cast("double"), 0.5).alias("mediana_sku")
)
df_razao = (
    positivos.join(mediana_sku, "sku")
    .withColumn("razao", F.col("preco_lista").cast("double") / F.col("mediana_sku"))
)

print("=== Distribuição da razão preço / mediana do SKU ===")
df_razao.select("razao").summary("min", "1%", "5%", "25%", "50%", "75%", "95%", "99%", "max").show()

# Contagem por faixa de razão: abaixo de 0,5x e acima de 2x são desvios grandes;
# acima de 10x é o que chamamos de erro óbvio (base do rótulo de apoio).
print("=== Contagem por faixa de razão ===")
df_razao.select(
    F.sum((F.col("razao") < 0.5).cast("int")).alias("abaixo_0_5x"),
    F.sum(((F.col("razao") >= 0.5) & (F.col("razao") <= 2)).cast("int")).alias("entre_0_5x_e_2x"),
    F.sum(((F.col("razao") > 2) & (F.col("razao") <= 10)).cast("int")).alias("entre_2x_e_10x"),
    F.sum((F.col("razao") > 10).cast("int")).alias("acima_10x"),
).show()

In [0]:
# ------------------------------------------------------------
# O preço muda ao longo do tempo?
# Compara o nível de preço entre micro-lotes, do primeiro ao último.
# Se houvesse tendência (inflação, promoções), a data seria uma feature útil.
# ATENÇÃO: esta célula cria normais e por_lote_razao, que as seguintes usam.
# Rode sempre com "Executar tudo".
# ------------------------------------------------------------
from pyspark.sql import Window

# 7.1 Como são os nomes dos arquivos (micro-lotes)?
# O nome carrega a data e a hora (ano/mês/dia/hora), então a ordem alfabética
# é também a ordem cronológica.
arquivos = df_prod.select("bronze_source_file").distinct()
print("=== Primeiros 5 nomes (ordem alfabética) ===")
arquivos.orderBy("bronze_source_file").show(5, truncate=False)
print("=== Últimos 5 nomes ===")
arquivos.orderBy(F.desc("bronze_source_file")).show(5, truncate=False)

# 7.2 Razão média por micro-lote.
# Só entram preços positivos até 10x a mediana do SKU ("normais"), para que os erros
# gigantes não distorçam a média. Se os preços subissem ou descessem com o tempo,
# a razão média mudaria de um lote para outro.
normais = df_razao.filter(F.col("razao") <= 10)
por_lote_razao = normais.groupBy("bronze_source_file").agg(
    F.avg("razao").alias("media_razao"),
    F.count("*").alias("linhas"),
)

print("=== Variação da razão média entre os micro-lotes ===")
por_lote_razao.agg(
    F.round(F.min("media_razao"), 3).alias("menor"),
    F.round(F.max("media_razao"), 3).alias("maior"),
    F.round(F.stddev("media_razao"), 3).alias("desvio"),
).show()

# 7.3 Juntando em 12 blocos de ~30 lotes, na ordem do nome do arquivo (ordem do tempo).
# Se não há tendência, a razão média fica parecida em todos os blocos.
ordem = Window.orderBy("bronze_source_file")
blocos = por_lote_razao.withColumn("bloco", F.ntile(12).over(ordem))
print("=== Razão média por bloco de tempo ===")
blocos.groupBy("bloco").agg(
    F.count("*").alias("lotes"),
    F.round(F.avg("media_razao"), 3).alias("razao_media"),
    F.min("bronze_source_file").alias("primeiro_arquivo"),
).orderBy("bloco").show(12, truncate=False)

In [0]:
# ------------------------------------------------------------
# O ruído do preço é aleatório ou segue um padrão?
# Testa se um preço alto (ou baixo) de um SKU é seguido por outro preço alto (ou baixo)
# na aparição seguinte do mesmo SKU, e mede o tamanho do ruído de cada SKU.
# Se o ruído fosse um padrão, um preço anterior serviria de feature.
# Usa a variável normais, criada na Célula 7.
# ------------------------------------------------------------
from pyspark.sql import Window

# 8.1 Correlação entre a razão atual e a da aparição anterior do mesmo SKU.
# A janela agrupa as linhas por SKU e as ordena pelo nome do arquivo, que segue a ordem
# do tempo. F.lag traz a razão da aparição imediatamente anterior do mesmo SKU.
# A primeira aparição de cada SKU não tem anterior e sai da conta.
# Correlação perto de 0 = o preço de agora não depende do anterior (ruído independente).
por_sku_tempo = Window.partitionBy("sku").orderBy("bronze_source_file")
df_lag = (
    normais
    .withColumn("razao_anterior", F.lag("razao").over(por_sku_tempo))
    .filter(F.col("razao_anterior").isNotNull())
)

print("=== Correlação com a aparição anterior do mesmo SKU ===")
df_lag.select(F.round(F.corr("razao", "razao_anterior"), 4).alias("correlacao")).show()

# 8.2 Tamanho do ruído dentro de cada SKU (desvio padrão da razão).
# Mostra se todos os SKUs oscilam o mesmo tanto ou se uns são mais "bagunçados" que outros.
# Se variar muito, uma régua única de alerta seria injusta, e o alerta precisa ser por grupo.
ruido_sku = normais.groupBy("sku").agg(F.stddev("razao").alias("desvio_razao"))

print("=== Desvio padrão da razão dentro de cada SKU (resumo entre os SKUs) ===")
ruido_sku.agg(
    F.round(F.min("desvio_razao"), 3).alias("menor"),
    F.round(F.avg("desvio_razao"), 3).alias("medio"),
    F.round(F.max("desvio_razao"), 3).alias("maior"),
).show()

In [0]:
# ------------------------------------------------------------
# Nível de preço por categoria raiz, unidade e formato de tamanho
# Monta uma tabela com uma linha por SKU (atributos + nível de preço) e compara
# o nível de preço entre categorias raiz e entre unidades. No fim, investiga o tamanho
# escrito no código do SKU.
# Usa mediana_sku (Célula 6). Cria df_sku, raizes e fora_padrao, que as
# células seguintes usam. Rode sempre com "Executar tudo".
# ------------------------------------------------------------

# Uma linha por SKU: atributos fixos + nível de preço (mediana).
# dropDuplicates(["sku"]) fica com uma linha por SKU; o join traz o nível de preço.
atributos = (
    df_prod
    .select("sku", "id_categoria", "id_categoria_pai", "unidade_medida", "nome_marca", "nome_produto")
    .dropDuplicates(["sku"])
)
df_sku = atributos.join(mediana_sku, "sku")
print("SKUs com nível de preço calculado:", df_sku.count())

# Nomes das categorias raiz. A tabela repete cada categoria 365 vezes (uma por micro-lote),
# então tiramos duplicatas. O id vira double para casar com id_categoria_pai da tabela de produtos.
raizes = (
    df_cat.filter(F.col("tipo_categoria") == "Categoria")
    .select(
        F.col("id_categoria").cast("double").alias("id_categoria_pai"),
        F.col("nome_categoria").alias("categoria_raiz"),
    )
    .dropDuplicates(["id_categoria_pai"])
)

# 9.1 Nível de preço por categoria raiz.
# Compara o preço típico (mediana dos SKUs) de cada categoria raiz.
print("=== Nível de preço dos SKUs por categoria raiz ===")
(
    df_sku.join(raizes, "id_categoria_pai", "left")
    .groupBy("categoria_raiz")
    .agg(
        F.count("*").alias("skus"),
        F.round(F.percentile_approx("mediana_sku", 0.5), 2).alias("mediana_dos_skus"),
        F.round(F.min("mediana_sku"), 2).alias("menor"),
        F.round(F.max("mediana_sku"), 2).alias("maior"),
    )
    .orderBy(F.desc("mediana_dos_skus"))
    .show(15, truncate=False)
)

# 9.2 Nível de preço por unidade de medida (kg, L, un).
print("=== Nível de preço dos SKUs por unidade ===")
(
    df_sku.groupBy("unidade_medida")
    .agg(
        F.count("*").alias("skus"),
        F.round(F.percentile_approx("mediana_sku", 0.5), 2).alias("mediana_dos_skus"),
        F.round(F.min("mediana_sku"), 2).alias("menor"),
        F.round(F.max("mediana_sku"), 2).alias("maior"),
    )
    .show()
)

# 9.3 Tamanho escondido no código do SKU (terceiro pedaço entre os hifens,
# ex.: "agua-bonafont-15l-natural" -> "15l"). É só uma investigação: no fim,
# o tamanho foi extraído do nome do produto, porque este código é ambíguo.
df_sku = df_sku.withColumn("token_tamanho", F.split("sku", "-").getItem(2))
print("Formatos de tamanho distintos:", df_sku.select("token_tamanho").distinct().count())
print("=== 25 formatos mais comuns ===")
df_sku.groupBy("token_tamanho").count().orderBy(F.desc("count")).show(25, truncate=False)

# SKUs cujo pedaço NÃO tem a forma "número + unidade" (ex.: 16un, 2kg, 05l).
# São os SKUs com código truncado.
fora_padrao = df_sku.filter(~F.col("token_tamanho").rlike(r"^\d+(un|kg|g|l|ml)$"))
print("SKUs cujo pedaço NÃO parece um tamanho:", fora_padrao.count())
fora_padrao.select("sku").show(10, truncate=False)

In [0]:
# ------------------------------------------------------------
# Os formatos de tamanho do código do SKU são ambíguos?
# Lista todos os formatos de tamanho que aparecem no código do SKU e compara
# alguns casos suspeitos com o nome do produto, que escreve o tamanho por extenso.
# Conclusão desta célula: o código do SKU não é confiável (ex.: "15l" é 1,5 L
# e "10kg" é 10 kg), então o tamanho será extraído do nome do produto.
# Usa df_sku e fora_padrao, criados na Célula 10.
# ------------------------------------------------------------

# 10.1 Todos os formatos de tamanho que existem (são 78, em ordem alfabética).
todos = [r["token_tamanho"] for r in df_sku.select("token_tamanho").distinct().orderBy("token_tamanho").collect()]
print("=== Todos os formatos ===")
print(todos)

# 10.2 Dois exemplos reais para cada formato ambíguo.
# Os pares 05kg/5kg, 01kg/1kg e 02kg/2kg parecem o mesmo tamanho escrito de dois
# jeitos, mas são diferentes: 05kg = 0,5 kg e 5kg = 5 kg. O zero à esquerda marca o decimal
# (005kg = 0,05 kg; 035l = 0,35 L), mas 15l também é decimal (1,5 L) e 10kg não é (10 kg).
# Só pelo código do SKU não dá para saber qual regra vale.
# Para cada formato, mostramos 2 SKUs com o nome completo e o nível de preço.
tokens_para_ver = ["05kg", "5kg", "01kg", "1kg", "02kg", "2kg", "005kg", "035l", "15l", "10kg", "1un", "50un"]
janela = Window.partitionBy("token_tamanho").orderBy("sku")
exemplos = (
    df_sku.filter(F.col("token_tamanho").isin(tokens_para_ver))
    .withColumn("ordem", F.row_number().over(janela))
    .filter(F.col("ordem") <= 2)
    .select(
        "token_tamanho",
        "nome_produto",
        "unidade_medida",
        F.round("mediana_sku", 2).alias("nivel_preco"),
    )
    .orderBy("token_tamanho")
)
print("\n=== Exemplos reais por formato ===")
exemplos.show(40, truncate=False)

# 10.3 Os SKUs que parecem cortados: o código termina no meio (ex.: "...-02k").
# São 4 SKUs, todos de pães especiais. No nome do produto o tamanho está completo.
print("=== SKUs fora do padrão, com o nome completo ===")
fora_padrao.select("sku", "nome_produto", "unidade_medida").show(truncate=False)

In [0]:
# ------------------------------------------------------------
# O tamanho do produto influencia o preço?
# Extrai o tamanho do nome do produto e mede, dentro de cada subcategoria, o quanto
# tamanhos maiores andam junto com preços maiores.
# Usa df_sku (Célula 10). Cria a coluna tamanho em df_sku, que as células seguintes usam.
# ------------------------------------------------------------

# 11.1 Extrair tamanho e unidade do nome do produto (ex.: "... 0.35L - Amstel").
# Como ler o padrão:
#   (\d+(?:\.\d+)?)  -> um número, com parte decimal opcional (35, 0.35, 1.5)
#   (kg|L|un)        -> a unidade
#   \s-\s            -> o " - " que separa o tamanho da marca
# Este é o mesmo padrão que o notebook 08 usa na engenharia de features.
padrao = r"(\d+(?:\.\d+)?)(kg|L|un)\s-\s"
numero = F.regexp_extract("nome_produto", padrao, 1)
unidade = F.regexp_extract("nome_produto", padrao, 2)

# F.when(... != "") evita converter texto vazio em número (daria erro no modo ANSI).
df_sku = (
    df_sku
    .withColumn("tamanho", F.when(numero != "", numero.cast("double")))
    .withColumn("unidade_no_nome", F.when(unidade != "", unidade))
)

# Conferência da extração: o esperado é 0 SKUs sem tamanho e 0 com unidade diferente.
print("=== Conferência da extração ===")
print("SKUs sem tamanho extraído:", df_sku.filter(F.col("tamanho").isNull()).count())
print("SKUs em que a unidade do nome é diferente de unidade_medida:",
      df_sku.filter(F.col("unidade_no_nome") != F.col("unidade_medida")).count())
df_sku.agg(F.min("tamanho").alias("menor_tamanho"), F.max("tamanho").alias("maior_tamanho")).show()
df_sku.select("nome_produto", "tamanho", "unidade_no_nome").orderBy(F.desc("tamanho")).show(5, truncate=False)

# 11.2 Correlação entre log(tamanho) e nível de preço, dentro de cada subcategoria.
# Correlação perto de +1: tamanhos maiores têm preço maior; perto de 0: o tamanho não
# muda o preço. Usamos o logaritmo do tamanho porque dobrar o tamanho (de 1 para 2 kg)
# pesa tanto quanto dobrar de 10 para 20 kg.
subcategorias = (
    df_cat.filter(F.col("tipo_categoria") == "Subcategoria")
    .select("id_categoria", F.col("nome_categoria").alias("subcategoria"))
    .dropDuplicates(["id_categoria"])
)

# Só entram grupos (subcategoria + unidade) com pelo menos 8 SKUs e 3 tamanhos distintos:
# com menos que isso, a correlação seria instável. tamanho > 0 é necessário para o log.
com_tamanho = df_sku.filter(F.col("tamanho") > 0).withColumn("log_tamanho", F.log("tamanho"))
corr_sub = (
    com_tamanho.groupBy("id_categoria", "unidade_medida")
    .agg(
        F.count("*").alias("skus"),
        F.countDistinct("tamanho").alias("tamanhos_distintos"),
        F.corr("log_tamanho", "mediana_sku").alias("corr"),
    )
    .filter((F.col("skus") >= 8) & (F.col("tamanhos_distintos") >= 3))
)

print("=== Resumo das correlações (subcategorias com pelo menos 8 SKUs e 3 tamanhos) ===")
corr_sub.agg(
    F.count("*").alias("grupos"),
    F.round(F.avg("corr"), 3).alias("corr_media"),
    F.round(F.percentile_approx("corr", 0.5), 3).alias("corr_mediana"),
    F.sum((F.col("corr") > 0.5).cast("int")).alias("acima_de_0_5"),
    F.sum((F.col("corr") < -0.5).cast("int")).alias("abaixo_de_menos_0_5"),
).show()

# Onde o tamanho mais pesa e onde menos pesa.
corr_nome = corr_sub.join(subcategorias, "id_categoria", "left")
print("=== 8 subcategorias onde o tamanho MAIS pesa ===")
corr_nome.select("subcategoria", "skus", F.round("corr", 3).alias("corr")).orderBy(F.desc("corr")).show(8, truncate=False)
print("=== 5 subcategorias onde o tamanho MENOS pesa ===")
corr_nome.select("subcategoria", "skus", F.round("corr", 3).alias("corr")).orderBy("corr").show(5, truncate=False)

In [0]:
# ------------------------------------------------------------
# A marca influencia o preço?
# Compara cada SKU com os "vizinhos" de mesma subcategoria e mesmo tamanho. Se alguma
# marca fosse sempre mais cara (ou mais barata) que os vizinhos, a marca explicaria preço.
# Também testa se Genérico e Marca Própria têm desconto.
# Usa df_sku com a coluna tamanho (Célula 12). Cria df_rel.
# ------------------------------------------------------------

# 12.1 Quantos SKUs cada marca tem.
# Marcas com poucos SKUs dão pouca informação para medir efeito.
por_marca = df_sku.groupBy("nome_marca").agg(F.count("*").alias("skus"))
print("Marcas distintas entre os SKUs:", por_marca.count())
por_marca.agg(
    F.min("skus").alias("minimo"),
    F.percentile_approx("skus", 0.5).alias("mediana"),
    F.max("skus").alias("maximo"),
).show()
print("=== 10 marcas com mais SKUs ===")
por_marca.orderBy(F.desc("skus")).show(10, truncate=False)

# 12.2 Nível do SKU em relação aos vizinhos (mesma subcategoria e mesmo tamanho).
# rel = nível do SKU / nível típico (mediana) dos vizinhos. rel = 1 é igual aos vizinhos;
# 1,10 é 10% mais caro. Só entram grupos com pelo menos 4 SKUs, para a mediana ser confiável.
# Isolar subcategoria e tamanho tira o efeito deles e deixa só o que sobra:
# diferenças entre SKUs e entre marcas.
celulas = df_sku.groupBy("id_categoria", "tamanho").agg(
    F.count("*").alias("vizinhos"),
    F.percentile_approx("mediana_sku", 0.5).alias("mediana_celula"),
)
df_rel = (
    df_sku.join(celulas, ["id_categoria", "tamanho"])
    .filter(F.col("vizinhos") >= 4)
    .withColumn("rel", F.col("mediana_sku") / F.col("mediana_celula"))
)

print("=== Quanto os SKUs de mesma subcategoria e tamanho diferem entre si ===")
df_rel.agg(
    F.count("*").alias("skus_comparados"),
    F.round(F.avg("rel"), 3).alias("rel_media"),
    F.round(F.stddev("rel"), 3).alias("rel_desvio"),
    F.round(F.min("rel"), 2).alias("menor"),
    F.round(F.max("rel"), 2).alias("maior"),
).show()

# 12.3 Efeito da marca (marcas com pelo menos 10 SKUs comparados).
# Média de rel por marca. Se a marca importasse, algumas teriam média bem acima ou abaixo de 1.
# Mas, com ~10 SKUs por marca e ~10% de variação entre SKUs, o acaso sozinho já produz
# diferenças de ~3% entre marcas. Diferenças menores que isso não indicam efeito real.
efeito_marca = (
    df_rel.groupBy("nome_marca")
    .agg(F.count("*").alias("skus"), F.round(F.avg("rel"), 3).alias("rel_media"))
    .filter(F.col("skus") >= 10)
)
print("=== Quanto as marcas diferem entre si (rel_media por marca) ===")
efeito_marca.agg(
    F.count("*").alias("marcas"),
    F.round(F.min("rel_media"), 3).alias("menor"),
    F.round(F.max("rel_media"), 3).alias("maior"),
    F.round(F.stddev("rel_media"), 3).alias("desvio"),
).show()
print("=== 8 marcas mais caras que os vizinhos ===")
efeito_marca.orderBy(F.desc("rel_media")).show(8, truncate=False)
print("=== 8 marcas mais baratas que os vizinhos ===")
efeito_marca.orderBy("rel_media").show(8, truncate=False)

# 12.4 Genérico e Marca Própria: têm desconto em relação aos vizinhos?
# rel_media perto de 1 = sem desconto.
print("=== Genérico e Marca Própria ===")
(
    df_rel.filter(F.col("nome_marca").isin("Genérico", "Marca Própria"))
    .groupBy("nome_marca")
    .agg(F.count("*").alias("skus"), F.round(F.avg("rel"), 3).alias("rel_media"))
    .show()
)

In [0]:
# ------------------------------------------------------------
# Escadinha de baselines e piso de erro
# Mede o erro de quatro "chutes" de preço, do mais simples ao mais preciso, para saber
# o que é um bom resultado para o modelo. O MAE médio mostra quanto cada chute erra,
# em reais, e o erro relativo mostra quanto isso pesa em relação ao preço previsto.
# Usa normais (Célula 7) e df_sku com tamanho (Célula 12).
# ------------------------------------------------------------

# Base: preços positivos "normais" (até 10x a mediana do SKU), com subcategoria,
# tamanho e mediana do SKU. São 285.449 linhas, as mesmas que entram no treino.
base = (
    normais
    .select("sku", "id_categoria", "preco_lista", "mediana_sku")
    .join(df_sku.select("sku", "tamanho"), "sku")
    .withColumn("preco", F.col("preco_lista").cast("double"))
)

# Quatro previsores, do mais simples ao mais preciso:
#   1. pred_global:          a mediana de todos os preços, para qualquer produto
#   2. pred_subcategoria:    a mediana dos preços da subcategoria
#   3. pred_subcat_tamanho:  a mediana dos preços da subcategoria e do mesmo tamanho
#   4. pred_sku:             a mediana do próprio SKU (o piso: nem um modelo perfeito
#                            consegue ir muito além, porque o resto é ruído)
mediana_global = base.approxQuantile("preco", [0.5], 0.001)[0]

med_sub = base.groupBy("id_categoria").agg(
    F.percentile_approx("preco", 0.5).alias("pred_subcategoria")
)
med_cel = base.groupBy("id_categoria", "tamanho").agg(
    F.percentile_approx("preco", 0.5).alias("pred_subcat_tamanho")
)

b = (
    base
    .join(med_sub, "id_categoria")
    .join(med_cel, ["id_categoria", "tamanho"])
    .withColumn("pred_global", F.lit(mediana_global))
    .withColumn("pred_sku", F.col("mediana_sku"))
)

print("Linhas avaliadas:", b.count())
print(f"Mediana global usada como chute: R$ {mediana_global:.2f}\n")

# MAE = média de |preço real - previsão| (em R$).
# Erro relativo = média de |preço real - previsão| / previsão (em %).
# ATENÇÃO: estes valores usam todas as linhas, sem separar treino e teste, e por isso são
# um pouco otimistas. Na modelagem, o mesmo cálculo deve ser refeito no fold de teste,
# com as medianas aprendidas só no treino.
print(f"{'Previsor':<22}{'MAE (R$)':>10}{'Erro relativo (%)':>20}")

for nome in ["pred_global", "pred_subcategoria", "pred_subcat_tamanho", "pred_sku"]:
    r = b.agg(
        F.avg(F.abs(F.col("preco") - F.col(nome))).alias("mae"),
        F.avg(F.abs(F.col("preco") - F.col(nome)) / F.col(nome)).alias("rel"),
    ).collect()[0]
    print(f"{nome:<22}{r['mae']:>10.2f}{r['rel'] * 100:>20.1f}")

%md
# Conclusões da EDA — P3 Preços quebrados no catálogo

## 1. O que são os dados

- Silver `ecommerce_produtos`: 292.000 linhas = 365 micro-lotes × 800 linhas, chegados a cada ~2 min, em janelas de coleta entre 01/10 e 15/10/2026 (há intervalos sem dados entre elas; fluxo "real-time").
- 3.805 SKUs distintos, cada um aparece 42–123 vezes e nunca duas vezes no mesmo lote. Grão: (micro-lote, SKU) = uma observação de preço.
- Sem nulos. Categoria e marca são fixas por SKU. 12 categorias raiz, 123 subcategorias, 239 marcas.
- `preco_custo` não existe nos dados ingeridos → a margem fica fora da entrega (decisão do Geovany, 09/11/2026).
- `descricao` é texto genérico repetido → descartada.
- O tamanho está escrito no `nome_produto` (ex.: "Água Mineral Natural 1.5L - Bonafont") e foi extraído de lá. O código do SKU é ambíguo (15l = 1,5 L, 10kg = 10 kg) e 4 SKUs vêm truncados.

## 2. Como é o preço

- Média R$ 35,68 (puxada pelos erros) e mediana R$ 19,23; mínimo −R$ 299,99 e máximo R$ 14.999,50.
- Erros visíveis: 2.258 preços zero + 2.113 negativos (= 4.371, regra 02, 1,50%); 2.180 preços acima de 10× a mediana do próprio SKU; 1.129 entre 2× e 10× (zona cinzenta).
- A regra 07 (limite ≈ R$ 5.000) pega só 72 linhas: deixa passar os preços gigantes abaixo do limite. O rótulo fraco (regras 02 e 07) é incompleto.
- A regra 09 mede vendas, não preço: não usar como rótulo do P3.
- Preço normal = nível do SKU × ruído independente. 90% dos preços ficam entre 0,37× e 1,69× da mediana do SKU. Correlação com a aparição anterior = 0,018. Sem tendência no tempo (razão média por lote entre 0,96 e 1,06). O ruído varia entre SKUs (desvio de 0,19 a 0,80).

## 3. O que explica o nível de preço de um SKU

- Subcategoria: maior ganho (MAE R$ 14,81 → 10,19).
- Tamanho: ganho adicional (10,19 → 9,23). Pesa muito em 20 de 49 subcategorias analisadas (Lenços Umedecidos, Rações, Fraldas, Areia Sanitária, Água Sanitária, Sacos de Lixo, Água Mineral), com correlação 0,84–0,94, e quase nada em outras (Molhos, Torradas, Balas). O efeito depende da subcategoria.
- Categoria raiz: só Mundo Bebê e Pets têm nível claramente mais alto.
- Marca, Genérico/Marca Própria: sem efeito mensurável (±2%, menos que o acaso).
- Unidade (kg/L/un) e tempo: irrelevantes.
- Piso de erro: mesmo conhecendo o SKU, o MAE continua em R$ 9,04 (34,4%); só com subcategoria + tamanho é R$ 9,23 (34,7%).

## 4. Decisões para a Engenharia de Features e o modelo

- Alvo: `preco_lista`. Treinar com preços > 0 e sem os gigantes óbvios (ou usar objetivo robusto, tipo L1/Huber).
- Features: subcategoria, categoria raiz, tamanho, log_tamanho, unidade, tamanho relativo dentro da subcategoria, variante do produto (texto do nome), marca (testar com e sem), flag Genérico/Marca Própria.
- Fora: data/hora, preço anterior, `descricao`, regra 09.
- Alerta: resíduo robusto (MAD) por grupo. O limiar tem que ficar bem acima do ruído de ~±35%.
- Avaliação: comparar com a escadinha de baselines (global, subcategoria, subcategoria + tamanho e o piso, a mediana do próprio SKU). Medir Precision/Recall contra o rótulo fraco e contra um rótulo de apoio (preço ≤ 0 ou > 10× a mediana do SKU), usado só para avaliar.
- Validação: como as features quase identificam o SKU, separar treino e teste por SKU (ou por tempo), e não por linha sorteada.
- Expectativa honesta: o MAE do LightGBM deve ficar perto de R$ 9,2. O valor do modelo está no alerta, e não em prever cada preço.

## 5. Limitações e decisões (09/11/2026)

- Sem `preco_custo`, não há cruzamento com margem: a margem fica fora da entrega.
- Não há metas numéricas de sucesso para o P3: reportamos as métricas e comparamos com os baselines.
- O rótulo de apoio (preço ≤ 0 ou > 10× a mediana do SKU) foi aprovado e é usado só na avaliação.   
